In [ ]:
# =============================================
# ✅ CLEAN & FINAL LayoutLMv3 on CORD-v1
# Warnings suppressed + Optimized
# =============================================

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message="Some weights of LayoutLMv3ForTokenClassification")

import json
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    LayoutLMv3ForTokenClassification,
    LayoutLMv3Processor,
    TrainingArguments,
    Trainer,
)
from seqeval.metrics import precision_score, recall_score, f1_score

print("✅ Dependencies and imports loaded\n")

# 1. Load Dataset
dataset = load_dataset("naver-clova-ix/cord-v1", split="train[:200]")

# 2. Labels
labels = [
    "O", "B-TOTAL", "I-TOTAL", "B-DATE", "I-DATE", "B-VENDOR", "I-VENDOR",
    "B-RECEIPT_ID", "I-RECEIPT_ID", "B-MENU", "I-MENU", "B-PRICE", "I-PRICE"
]
label2id = {label: idx for idx, label in enumerate(labels)}
id2label = {idx: label for label, idx in label2id.items()}

# 3. Processor + Model
processor = LayoutLMv3Processor.from_pretrained("microsoft/layoutlmv3-base", apply_ocr=False)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    "microsoft/layoutlmv3-base",
    num_labels=len(labels),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True   # Suppresses weight initialization warning
)

print("✅ Model & Processor loaded\n")

# 4. Preprocessing
def safe_split(text):
    return text.strip().split() if isinstance(text, str) and text.strip() else []

def extract_data(example):
    image = example["image"]
    try:
        gt = json.loads(example["ground_truth"])
        parse = gt.get("gt_parse", {})
    except:
        parse = {}

    words, ner_tags, bboxes = [], [], []

    for item in parse.get("menu", []):
        if isinstance(item, dict):
            for key, tag in [("nm", "MENU"), ("price", "PRICE")]:
                val = item.get(key)
                if isinstance(val, str) and val.strip():
                    for i, w in enumerate(safe_split(val)):
                        words.append(w)
                        ner_tags.append(label2id[f"B-{tag}"] if i == 0 else label2id[f"I-{tag}"])
                        bboxes.append([0, 0, 999, 999])

    for key, tag in [("vendor", "VENDOR"), ("date", "DATE"), ("total", "TOTAL")]:
        val = parse.get(key)
        if isinstance(val, str) and val.strip():
            for i, w in enumerate(safe_split(val)):
                words.append(w)
                ner_tags.append(label2id[f"B-{tag}"] if i == 0 else label2id[f"I-{tag}"])
                bboxes.append([0, 0, 999, 999])

    if not words:
        words = ["receipt"]
        ner_tags = [label2id["O"]]
        bboxes = [[0, 0, 999, 999]]

    return {"words": words, "ner_tags": ner_tags, "bboxes": bboxes, "image": image}

def preprocess(example):
    data = extract_data(example)
    encoding = processor(
        data["image"],
        data["words"],
        boxes=data["bboxes"],
        word_labels=data["ner_tags"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )
    # Remove extra batch dimension
    encoding = {k: v.squeeze(0) for k, v in encoding.items()}
    return encoding

# Tokenize
tokenized_dataset = dataset.map(
    preprocess,
    remove_columns=dataset.column_names,
    desc="Processing LayoutLMv3 data"
)

tokenized_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "bbox", "labels", "pixel_values"])

print(f"✅ Tokenization completed! pixel_values shape: {tokenized_dataset[0]['pixel_values'].shape}\n")

# Split
split = tokenized_dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = split["train"]
eval_dataset = split["test"]

# Collator
class LayoutLMv3Collator:
    def __call__(self, features):
        batch = {}
        for k in features[0].keys():
            batch[k] = torch.stack([f[k] for f in features])
        return batch

data_collator = LayoutLMv3Collator()

# Metrics
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=2)
    true_preds = [[id2label[p] for p, l in zip(pred, lab) if l != -100]
                  for pred, lab in zip(preds, labels)]
    true_labs = [[id2label[l] for l in lab if l != -100] for lab in labels]

    return {
        "precision": precision_score(true_labs, true_preds, zero_division=0),
        "recall": recall_score(true_labs, true_preds, zero_division=0),
        "f1": f1_score(true_labs, true_preds, zero_division=0),
    }

# Training Arguments
training_args = TrainingArguments(
    output_dir="./cord-layoutlmv3-final",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=3,
    fp16=True,
    weight_decay=0.01,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    logging_steps=10,
    disable_tqdm=False,
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

print("🚀 Starting training...\n")
trainer.train()

# Final Evaluation
metrics = trainer.evaluate()
print("\n📊 Final Evaluation Results:")
for k, v in metrics.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

# Save
trainer.save_model("./cord-layoutlmv3-final")
processor.save_pretrained("./cord-layoutlmv3-final")
print("\n✅ Model saved successfully! (No warnings)")

✅ Dependencies and imports loaded



Some weights of LayoutLMv3ForTokenClassification were not initialized from the model checkpoint at microsoft/layoutlmv3-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


✅ Model & Processor loaded



Processing LayoutLMv3 data:   0%|          | 0/200 [00:00<?, ? examples/s]

✅ Tokenization completed! pixel_values shape: torch.Size([3, 224, 224])

🚀 Starting training...



Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.380700,0.206682,0.946565,0.898551,0.921933
2,0.126600,0.084027,0.948529,0.934783,0.941606
3,0.057600,0.057688,0.957143,0.971014,0.964029



📊 Final Evaluation Results:
  eval_loss: 0.0577
  eval_precision: 0.9571
  eval_recall: 0.9710
  eval_f1: 0.9640
  eval_runtime: 2.5452
  eval_samples_per_second: 15.7160
  eval_steps_per_second: 3.9290
  epoch: 3.0000

✅ Model saved successfully! (No warnings)


In [ ]:
import torch
from PIL import Image
from transformers import LayoutLMv3Processor, LayoutLMv3ForTokenClassification

# Load your trained model
model_path = "./cord-layoutlmv3-final"

processor = LayoutLMv3Processor.from_pretrained(model_path, apply_ocr=False)
model = LayoutLMv3ForTokenClassification.from_pretrained(model_path)

model.eval()
print("✅ Model loaded for inference\n")

def predict_receipt(image_path):
    image = Image.open(image_path).convert("RGB")

    # For real usage, you should do proper OCR + bounding boxes.
    # Here is a simple placeholder version:
    words = ["Sample", "Receipt", "Total", "1000"]
    boxes = [[0,0,999,999]] * len(words)   # placeholder

    encoding = processor(
        image,
        words,
        boxes=boxes,
        truncation=True,
        padding="max_length",
        max_length=512,
        return_tensors="pt"
    )

    with torch.no_grad():
        outputs = model(**encoding)

    predictions = torch.argmax(outputs.logits, dim=2)

    # Decode
    id2label = model.config.id2label
    predicted_labels = [id2label[p.item()] for p in predictions[0]]

    print("Predicted Entities:")
    for word, label in zip(words, predicted_labels):
        if label != "O":
            print(f"{word:15} → {label}")

# Example usage:
# predict_receipt("/content/test_receipt.jpg")

✅ Model loaded for inference



✅ Model loaded for testing

✅ Loaded 100 unseen samples for testing

🚀 Starting evaluation on dataset samples...


🔹 Test Sample #1
------------------------------------------------------------
Word                 Predicted Label     
------------------------------------------------------------
NASI                 O                   
PUTIH                B-MENU              
6.000                I-MENU              
BASO                 I-MENU              
KUAH                 I-MENU              
43,636               I-MENU              
{'total_price':      I-MENU              
'54,600',            B-PRICE             
'cashprice':         I-MENU              
'54.600',            B-PRICE             
'changeprice':       B-MENU              
'0'}                 I-MENU              
------------------------------------------------------------

🔹 Test Sample #2
------------------------------------------------------------
Word                 Predicted Label     
------------------

In [ ]:
# =============================================
# ✅ FIXED ONNX Export for LayoutLMv3
# Using latest Optimum method
# =============================================

!pip install --quiet --upgrade optimum onnxruntime

import warnings
warnings.filterwarnings("ignore")

from optimum.onnxruntime import ORTModelForTokenClassification
from transformers import LayoutLMv3Processor
from optimum.exporters.onnx import export, main_export

print("🚀 Starting ONNX export with latest method...\n")

model_path = "./cord-layoutlmv3-final"
onnx_save_dir = "./cord-layoutlmv3-onnx"

# Method 1: Using CLI (Simplest and Most Reliable)
print("Exporting using optimum-cli...")
!optimum-cli export onnx \
    --model {model_path} \
    --task token-classification \
    --opset 14 \
    --device cpu \
    {onnx_save_dir}

print(f"\n✅ ONNX model exported successfully to: {onnx_save_dir}")

# Also copy the processor
processor = LayoutLMv3Processor.from_pretrained(model_path, apply_ocr=False)
processor.save_pretrained(onnx_save_dir)

print("✅ Processor files saved!")
print("\nONNX files created:")
!ls -lh {onnx_save_dir}

🚀 Starting ONNX export with latest method...

Exporting using optimum-cli...
2026-03-29 03:45:41.670453: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774755941.754555   12352 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774755941.778754   12352 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774755941.846416   12352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774755941.848717   12352 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:0

In [ ]:
# Test the exported ONNX model
from optimum.onnxruntime import ORTModelForTokenClassification
from transformers import LayoutLMv3Processor
import torch

model = ORTModelForTokenClassification.from_pretrained(onnx_save_dir)
processor = LayoutLMv3Processor.from_pretrained(onnx_save_dir, apply_ocr=False)

print("✅ ONNX Model loaded successfully!")
print(f"Model type: {type(model)}")

✅ ONNX Model loaded successfully!
Model type: <class 'optimum.onnxruntime.modeling.ORTModelForTokenClassification'>


In [ ]:
from huggingface_hub import notebook_login
from optimum.onnxruntime import ORTModelForTokenClassification
from transformers import LayoutLMv3Processor

notebook_login()

save_dir = "./cord-layoutlmv3-onnx"
repo_id = "ShaibinkB/cord-layoutlmv3-onnx"

model = ORTModelForTokenClassification.from_pretrained(save_dir)
processor = LayoutLMv3Processor.from_pretrained(save_dir, apply_ocr=False)

# Optimum ONNX model: first arg = local folder, then repository_id
model.push_to_hub(save_dir, repository_id=repo_id)

# Processor can be pushed in the usual way
processor.push_to_hub(repo_id)

print("✅ Successfully pushed to Hugging Face Hub")

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...ayoutlmv3-onnx/model.onnx:   0%|          |  554kB /  504MB            

✅ Successfully pushed to Hugging Face Hub
